# LiteVoiceNet — Training Notebook\n\n**Step 1** — Drone microphone array project.  \nThesis: *Human Voice Signal Identification System using Drone-Mounted Microphone Arrays*.\n\nTrains `LiteVoiceNet` on **synthetic data** and exports ONNX + TFLite INT8.  \nUse **Kernel → Restart &amp; Run All**. CUDA is required — Cell 1 raises if unavailable.\n\n| Cell | Task |\n|------|------|\n| 1 | Environment check (CUDA) |\n| 2 | Config display |\n| 3 | Synthetic dataset + visualisation |\n| 4 | Model + parameter count + FLOPs |\n| 5 | Training setup |\n| 6 | Training loop (GPU) |\n| 7 | Loss curves |\n| 8 | Spectrogram + mask visualisation |\n| 9 | KWS confusion matrix |\n| 10 | VAD ROC / PR curves |\n| 11 | ONNX float32 export |\n| 12 | ONNX INT8 + TFLite export |\n| 13 | CPU latency benchmark |

In [1]:
# Cell 1 : Environment check\nimport sys, os\nfrom pathlib import Path\nimport warnings\nwarnings.filterwarnings('ignore')\n\nROOT = Path('..').resolve()\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nimport torch\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nprint('Python  :', sys.version.split()[0])\nprint('PyTorch :', torch.__version__)\nprint('NumPy   :', np.__version__)\nprint('CUDA    :', torch.cuda.is_available())\n\nif not torch.cuda.is_available():\n    sep = '!' * 62\n    print()\n    print(sep)\n    print('  CUDA NOT AVAILABLE')\n    print('  Training on CPU would be impractically slow.')\n    print('  Install PyTorch with CUDA 12.4:')\n    print('    pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu124')\n    print('  Then verify: torch.cuda.is_available() == True')\n    print(sep)\n    raise RuntimeError('CUDA not found. Fix your environment and re-run this cell.')\n\ngpu = torch.cuda.get_device_properties(0)\nprint('GPU     :', gpu.name)\nprint('VRAM    : {:.1f} GB'.format(gpu.total_memory / 1e9))\ndevice = torch.device('cuda')\nprint()\nprint('Using device:', device)

In [2]:
# Cell 2 : Configuration\nfrom config import CFG\n\nhop_ms = CFG.signal.hop_size / CFG.signal.sample_rate_raw * 1000\nseq_ms = CFG.synth.seq_frames * hop_ms\n\nprint('=== Signal ===')\nprint('  Sample rate : {} Hz'.format(CFG.signal.sample_rate_raw))\nprint('  n_fft       : {}  (frame {:.2f} ms)'.format(\n    CFG.signal.n_fft, CFG.signal.n_fft / CFG.signal.sample_rate_raw * 1000))\nprint('  hop         : {}  ({:.2f} ms)'.format(CFG.signal.hop_size, hop_ms))\nprint('  freq bins   : {}'.format(CFG.signal.n_freq_bins))\nprint('=== Array ===')\nprint('  Active mics : {}'.format(CFG.array.num_active_mics))\nprint('  IPD pairs   : {}'.format(CFG.array.mic_pairs))\nprint('  Features    : {}  (1 log-mag + 4 IPD)'.format(CFG.array.num_features))\nprint('=== Model ===')\nprint('  Encoder ch  : {}'.format(CFG.model.enc_ch))\nprint('  GRU         : hidden={} x{} causal'.format(\n    CFG.model.gru_hidden, CFG.model.gru_layers))\nprint('  KWS classes : {}'.format(CFG.model.kws_classes))\nprint('=== Training ===')\nprint('  Epochs      : {}'.format(CFG.train.num_epochs))\nprint('  Batch size  : {}'.format(CFG.train.batch_size))\nprint('  LR          : {}'.format(CFG.train.learning_rate))\nprint('=== Synthetic data ===')\nprint('  Train/Val   : {} / {}'.format(CFG.synth.train_samples, CFG.synth.val_samples))\nprint('  Seq length  : {} frames  ({:.0f} ms)'.format(CFG.synth.seq_frames, seq_ms))

In [3]:
# Cell 3 : Synthetic dataset + visualisation\nfrom torch.utils.data import DataLoader\nfrom data import build_datasets\n\ntorch.manual_seed(CFG.train.seed)\nnp.random.seed(CFG.train.seed)\n\ntrain_ds, val_ds = build_datasets(CFG)\ntrain_loader = DataLoader(\n    train_ds, batch_size=CFG.train.batch_size, shuffle=True,\n    num_workers=CFG.train.num_workers, pin_memory=CFG.train.pin_memory)\nval_loader = DataLoader(\n    val_ds, batch_size=CFG.train.batch_size, shuffle=False,\n    num_workers=CFG.train.num_workers, pin_memory=CFG.train.pin_memory)\n\nprint('Train batches : {}  ({} samples)'.format(len(train_loader), len(train_ds)))\nprint('Val   batches : {}  ({} samples)'.format(len(val_loader),   len(val_ds)))\n\nfeat, mask_t, vad_t, kws_t = train_ds[0]\nprint('features={} mask={} vad={} kws={} ({})'.format(\n    tuple(feat.shape), tuple(mask_t.shape), tuple(vad_t.shape),\n    kws_t.item(), CFG.model.kws_classes[kws_t.item()]))\n\nfig, axes = plt.subplots(1, 3, figsize=(14, 3.5))\nim0 = axes[0].imshow(feat[0].numpy().T, aspect='auto', origin='lower', cmap='magma')\naxes[0].set_title('Log-magnitude (ref mic)')\naxes[0].set_xlabel('Time frames'); axes[0].set_ylabel('Freq bins')\nplt.colorbar(im0, ax=axes[0])\nim1 = axes[1].imshow(mask_t.numpy().T, aspect='auto', origin='lower',\n                     cmap='RdYlGn', vmin=0, vmax=1)\naxes[1].set_title('IRM target'); axes[1].set_xlabel('Time frames')\nplt.colorbar(im1, ax=axes[1])\naxes[2].plot(vad_t.numpy())\naxes[2].set_title('VAD target'); axes[2].set_xlabel('Time frames')\naxes[2].set_ylim(-0.1, 1.1); axes[2].grid(alpha=0.3)\nplt.suptitle('Synthetic sample', fontsize=12)\nplt.tight_layout(); plt.show()

In [4]:
# Cell 4 : Model + parameter count\nfrom model import build_model\n\nmodel = build_model(CFG).to(device)\ntotal_params     = sum(p.numel() for p in model.parameters())\ntrainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\nBUDGET = 1_500_000\n\nprint('Total parameters     : {:,}'.format(total_params))\nprint('Trainable parameters : {:,}'.format(trainable_params))\nassert total_params < BUDGET, 'Exceeds 1.5 M: {:,}'.format(total_params)\nprint('Budget used          : {:.1f} %'.format(total_params / BUDGET * 100))\n\ntry:\n    from torchinfo import summary as tinfo_summary\n    dummy = torch.zeros(1, CFG.array.num_features,\n                        CFG.synth.seq_frames, CFG.signal.n_freq_bins, device=device)\n    tinfo_summary(model, input_data=dummy, depth=2,\n                  col_names=['input_size', 'output_size', 'num_params'])\nexcept ImportError:\n    print('torchinfo not installed -- pip install torchinfo')\n\nmodel.eval()\nwith torch.no_grad():\n    dummy = torch.zeros(1, CFG.array.num_features,\n                        CFG.synth.seq_frames, CFG.signal.n_freq_bins, device=device)\n    mask_o, vad_o, kws_o, h_o = model(dummy)\nprint('Outputs: mask={} vad={} kws={} h_out={}'.format(\n    tuple(mask_o.shape), tuple(vad_o.shape),\n    tuple(kws_o.shape),  tuple(h_o.shape)))

In [5]:
# Cell 5 : Training setup\nfrom losses import build_loss\n\ncriterion = build_loss(CFG)\noptimizer = torch.optim.Adam(\n    model.parameters(), lr=CFG.train.learning_rate,\n    weight_decay=CFG.train.weight_decay)\nscheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n    optimizer, T_max=CFG.train.num_epochs, eta_min=1e-5)\n\nprint('Optimizer  :', type(optimizer).__name__)\nprint('Scheduler  :', type(scheduler).__name__)\nprint('Loss weights mask={:.2f} vad={:.2f} kws={:.2f}'.format(\n    CFG.loss.w_mask, CFG.loss.w_vad, CFG.loss.w_kws))

In [6]:
# Cell 6 : Training loop (GPU)\nimport time\n\nhistory = {\n    'train_total': [], 'train_mask': [], 'train_vad': [], 'train_kws': [],\n    'val_total':   [], 'val_mask':   [], 'val_vad':   [], 'val_kws':   [],\n}\nEPOCHS   = CFG.train.num_epochs\nLOG_FREQ = max(1, EPOCHS // 10)\nt_start  = time.time()\n\nfor epoch in range(1, EPOCHS + 1):\n    model.train()\n    t_tot = t_mask = t_vad = t_kws = 0.0; n_tr = 0\n    for feat, mask_tgt, vad_tgt, kws_tgt in train_loader:\n        feat     = feat.to(device, non_blocking=True)\n        mask_tgt = mask_tgt.to(device, non_blocking=True)\n        vad_tgt  = vad_tgt.to(device, non_blocking=True)\n        kws_tgt  = kws_tgt.to(device, non_blocking=True)\n        optimizer.zero_grad()\n        preds = model(feat)\n        loss, bd = criterion(preds, (mask_tgt, vad_tgt, kws_tgt))\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.train.grad_clip)\n        optimizer.step()\n        t_tot += loss.item(); t_mask += bd['mask']\n        t_vad += bd['vad'];   t_kws  += bd['kws']; n_tr += 1\n    scheduler.step()\n\n    model.eval()\n    v_tot = v_mask = v_vad = v_kws = 0.0; n_val = 0\n    with torch.no_grad():\n        for feat, mask_tgt, vad_tgt, kws_tgt in val_loader:\n            feat     = feat.to(device, non_blocking=True)\n            mask_tgt = mask_tgt.to(device, non_blocking=True)\n            vad_tgt  = vad_tgt.to(device, non_blocking=True)\n            kws_tgt  = kws_tgt.to(device, non_blocking=True)\n            preds    = model(feat)\n            loss, bd = criterion(preds, (mask_tgt, vad_tgt, kws_tgt))\n            v_tot += loss.item(); v_mask += bd['mask']\n            v_vad += bd['vad'];   v_kws  += bd['kws']; n_val += 1\n\n    history['train_total'].append(t_tot / n_tr)\n    history['train_mask'].append( t_mask / n_tr)\n    history['train_vad'].append(  t_vad  / n_tr)\n    history['train_kws'].append(  t_kws  / n_tr)\n    history['val_total'].append(  v_tot  / n_val)\n    history['val_mask'].append(   v_mask / n_val)\n    history['val_vad'].append(    v_vad  / n_val)\n    history['val_kws'].append(    v_kws  / n_val)\n\n    if epoch % LOG_FREQ == 0 or epoch == 1:\n        print('Ep {:3d}/{} | train {:.4f} (M:{:.3f} V:{:.3f} K:{:.3f}) | '\n              'val {:.4f} (M:{:.3f} V:{:.3f} K:{:.3f}) | {:.0f}s'.format(\n            epoch, EPOCHS, t_tot/n_tr, t_mask/n_tr, t_vad/n_tr, t_kws/n_tr,\n            v_tot/n_val, v_mask/n_val, v_vad/n_val, v_kws/n_val,\n            time.time() - t_start))\n\nprint('Training complete in {:.1f} s'.format(time.time() - t_start))

In [7]:
# Cell 7 : Loss curves\nfig, axes = plt.subplots(1, 4, figsize=(18, 3.5))\nfor ax, task in zip(axes, ['total', 'mask', 'vad', 'kws']):\n    ax.plot(history['train_' + task], label='train', color='tab:blue',   lw=2)\n    ax.plot(history['val_'   + task], label='val',   color='tab:orange', lw=2)\n    ax.set_title(task.upper() + ' loss')\n    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)\nfig.suptitle('LiteVoiceNet loss curves (synthetic data)', fontsize=13)\nplt.tight_layout(); plt.show()\nprint('Final  train: {:.4f}   val: {:.4f}'.format(\n    history['train_total'][-1], history['val_total'][-1]))

In [8]:
# Cell 8 : Mask visualisation\nmodel.eval()\nfeat_s, mask_gt_s, vad_gt_s, kws_gt_s = val_ds[0]\nwith torch.no_grad():\n    mask_pred, vad_pred, kws_pred, _ = model(feat_s.unsqueeze(0).to(device))\n\nmask_np = mask_pred[0, 0].cpu().numpy()\nmask_gt = mask_gt_s.numpy()\nlogmag  = feat_s[0].numpy()\n\nfig, axes = plt.subplots(1, 3, figsize=(15, 4))\naxes[0].imshow(logmag.T, aspect='auto', origin='lower', cmap='magma')\naxes[0].set_title('Log-magnitude input')\naxes[0].set_xlabel('Time'); axes[0].set_ylabel('Freq')\nim1 = axes[1].imshow(mask_gt.T, aspect='auto', origin='lower',\n                     cmap='RdYlGn', vmin=0, vmax=1)\naxes[1].set_title('IRM ground truth'); axes[1].set_xlabel('Time')\nplt.colorbar(im1, ax=axes[1])\nim2 = axes[2].imshow(mask_np.T, aspect='auto', origin='lower',\n                     cmap='RdYlGn', vmin=0, vmax=1)\naxes[2].set_title('IRM predicted'); axes[2].set_xlabel('Time')\nplt.colorbar(im2, ax=axes[2])\nplt.suptitle('Mask estimation (validation sample)', fontsize=12)\nplt.tight_layout(); plt.show()\n\nidx_pred = kws_pred[0].exp().argmax().item()\nprint('VAD  pred mean: {:.3f}  gt mean: {:.3f}'.format(\n    vad_pred[0,:,0].mean().item(), vad_gt_s.mean().item()))\nprint('KWS  pred: {}  gt: {}'.format(\n    CFG.model.kws_classes[idx_pred], CFG.model.kws_classes[kws_gt_s.item()]))

In [9]:
# Cell 9 : KWS confusion matrix\nfrom sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay\n\nmodel.eval()\nall_true_kws, all_pred_kws = [], []\nwith torch.no_grad():\n    for feat, _, _, kws_tgt in val_loader:\n        _, _, kws_out, _ = model(feat.to(device, non_blocking=True))\n        all_pred_kws.extend(kws_out.argmax(1).cpu().tolist())\n        all_true_kws.extend(kws_tgt.tolist())\n\ncm   = confusion_matrix(all_true_kws, all_pred_kws,\n                        labels=list(range(CFG.model.num_kws_classes)))\ndisp = ConfusionMatrixDisplay(cm, display_labels=CFG.model.kws_classes)\nfig, ax = plt.subplots(figsize=(6, 5))\ndisp.plot(ax=ax, colorbar=False, cmap='Blues')\nax.set_title('KWS confusion matrix (val, synthetic)')\nplt.tight_layout(); plt.show()\n\nacc = float((np.array(all_true_kws) == np.array(all_pred_kws)).mean())\nprint('KWS accuracy (synthetic): {:.3f}'.format(acc))\nprint('NOTE: use ManifestDataset with real speech for meaningful accuracy.')

In [10]:
# Cell 10 : VAD ROC / PR curves\nfrom sklearn.metrics import roc_curve, precision_recall_curve, auc\n\nmodel.eval()\nall_vad_true, all_vad_prob = [], []\nwith torch.no_grad():\n    for feat, _, vad_tgt, _ in val_loader:\n        _, vad_out, _, _ = model(feat.to(device, non_blocking=True))\n        all_vad_prob.extend(vad_out[:,:,0].cpu().numpy().ravel().tolist())\n        all_vad_true.extend(vad_tgt.numpy().ravel().tolist())\n\ny_true = np.array(all_vad_true); y_prob = np.array(all_vad_prob)\nn_pos  = int(y_true.sum()); n_neg = int((1 - y_true).sum())\nprint('VAD frames: {} positive, {} negative'.format(n_pos, n_neg))\n\nif n_pos > 0 and n_neg > 0:\n    fpr, tpr, _  = roc_curve(y_true, y_prob)\n    roc_auc      = auc(fpr, tpr)\n    prec, rec, _ = precision_recall_curve(y_true, y_prob)\n    pr_auc       = auc(rec, prec)\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    axes[0].plot(fpr, tpr, 'tab:blue', lw=2,\n                 label='ROC (AUC={:.3f})'.format(roc_auc))\n    axes[0].plot([0,1],[0,1],'k--',lw=1)\n    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')\n    axes[0].set_title('VAD ROC'); axes[0].legend(); axes[0].grid(alpha=0.3)\n    axes[1].plot(rec, prec, 'tab:orange', lw=2,\n                 label='PR (AUC={:.3f})'.format(pr_auc))\n    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')\n    axes[1].set_title('VAD Precision-Recall')\n    axes[1].legend(); axes[1].grid(alpha=0.3)\n    plt.suptitle('VAD evaluation (val, synthetic)', fontsize=12)\n    plt.tight_layout(); plt.show()\n    print('VAD ROC-AUC: {:.4f}   PR-AUC: {:.4f}'.format(roc_auc, pr_auc))\nelse:\n    print('WARNING: only one VAD class -- ROC/PR undefined.')\n    print('Increase synth.seq_frames or adjust the VAD threshold in data.py.')

In [11]:
# Cell 11 : ONNX float32 export\nimport export as exp_mod\nimport onnxruntime as ort\n\nos.makedirs(CFG.export_dir, exist_ok=True)\nonnx_path = os.path.join(CFG.export_dir, 'litevoicenet.onnx')\n\nmodel.eval(); model_cpu = model.cpu()\nonnx_out = exp_mod.export_onnx(model_cpu, CFG, output_path=onnx_path,\n                                seq_frames=CFG.synth.seq_frames)\n\nsess_opts = ort.SessionOptions()\nsess_opts.intra_op_num_threads = 1\nsess_fp32 = ort.InferenceSession(onnx_out, sess_options=sess_opts,\n                                  providers=['CPUExecutionProvider'])\nrng_t  = np.random.default_rng(0)\nx_test = rng_t.random((1, CFG.array.num_features,\n                        CFG.synth.seq_frames, CFG.signal.n_freq_bins)).astype(np.float32)\nhx_t   = np.zeros((CFG.model.gru_layers, 1, CFG.model.gru_hidden), np.float32)\nouts   = sess_fp32.run(None, {'features': x_test, 'h_in': hx_t})\nprint('ORT OK  mask={} vad={} kws={}'.format(\n    outs[0].shape, outs[1].shape, outs[2].shape))\nmodel = model.to(device)

In [12]:
# Cell 12 : INT8 quantisation\nimport os.path as osp\n\nint8_onnx_path = os.path.join(CFG.export_dir, 'litevoicenet_int8.onnx')\ntflite_path    = os.path.join(CFG.export_dir, 'litevoicenet_int8.tflite')\n\nint8_ok = exp_mod.quantize_onnx_int8(\n    onnx_out, output_path=int8_onnx_path, n_calib_samples=200, cfg=CFG)\ntflite_ok = exp_mod.export_tflite_int8(onnx_out, output_path=tflite_path)\n\ndef _fmtp(p, ok):\n    if ok and osp.exists(p):\n        return '{} ({:.1f} MB)'.format(p, osp.getsize(p) / 1e6)\n    return 'FAILED / SKIPPED'\n\nprint('=== Export summary ===')\nprint('  ONNX float32 :', _fmtp(onnx_out,       True))\nprint('  ONNX INT8    :', _fmtp(int8_onnx_path,  int8_ok))\nprint('  TFLite INT8  :', _fmtp(tflite_path,     tflite_ok))

In [13]:
# Cell 13 : CPU-only latency benchmark\nimport os.path as osp\n\nchunk_ms = CFG.synth.seq_frames * CFG.signal.hop_size / CFG.signal.sample_rate_raw * 1000\nprint('=' * 60)\nprint('CPU-only benchmark (1 thread -- simulates Pi 4 single core)')\nprint('Audio chunk: {:.1f} ms  |  Target: < 20 ms per chunk'.format(chunk_ms))\nprint('NOTE: final measurement must be done on real Pi 4 hardware.')\nprint('=' * 60)\n\nif osp.exists(onnx_path):\n    stats_fp32 = exp_mod.benchmark_cpu(onnx_out, CFG, n_runs=200, n_warmup=20)\n    sz32 = osp.getsize(onnx_out) / 1e6\n    rt   = stats_fp32['realtime_factor']\n    print()\n    print('Float32 ONNX ({:.1f} MB)'.format(sz32))\n    print('  Mean latency : {:.2f} ms'.format(stats_fp32['mean_ms']))\n    print('  P50  latency : {:.2f} ms'.format(stats_fp32['p50_ms']))\n    print('  P95  latency : {:.2f} ms'.format(stats_fp32['p95_ms']))\n    print('  RT factor    : {:.3f}x  ({})'.format(\n        rt, 'faster-than-RT' if rt < 1.0 else 'SLOWER-than-RT'))\n    print('  Budget 20ms  :', 'PASS' if stats_fp32['mean_ms'] < 20 else 'EXCEEDS (re-measure on Pi 4)')\n\nif int8_ok and osp.exists(int8_onnx_path):\n    stats_i8 = exp_mod.benchmark_cpu(int8_onnx_path, CFG, n_runs=200, n_warmup=20)\n    szi8     = osp.getsize(int8_onnx_path) / 1e6\n    speedup  = stats_fp32['mean_ms'] / stats_i8['mean_ms']\n    print()\n    print('INT8 ONNX ({:.1f} MB)'.format(szi8))\n    print('  Mean latency : {:.2f} ms'.format(stats_i8['mean_ms']))\n    print('  P95  latency : {:.2f} ms'.format(stats_i8['p95_ms']))\n    print('  Speedup      : {:.2f}x vs FP32'.format(speedup))\n    print('  Size ratio   : {:.1f}x smaller'.format(sz32 / szi8))\nelse:\n    print('INT8 benchmark skipped.')\n\nprint()\nprint('Models saved to:', osp.abspath(CFG.export_dir))